In [2]:
!unzip -q /content/funko-ml-main.zip -d /content/

In [3]:
import os

print("Project exists:", os.path.exists('/content/funko-ml-main'))
print("src exists:", os.path.exists('/content/funko-ml-main/src'))
print("evaluate.py exists:", os.path.exists('/content/funko-ml-main/src/evaluate.py'))
print("Processed data exists:", os.path.exists('/content/funko-ml-main/data/processed.npz'))
print("Folds exist:", os.path.exists('/content/funko-ml-main/data/folds.pkl'))

Project exists: True
src exists: True
evaluate.py exists: True
Processed data exists: True
Folds exist: True


In [8]:
import sys
import numpy as np
import pandas as pd

from sklearn.svm import SVC

SRC = '/content/funko-ml-main/src'

if SRC not in sys.path:
    sys.path.insert(0, SRC)

print("Source path added.")

Source path added.


In [10]:
import sys
import os
import importlib.util

# Exact location of evaluate.py
evaluate_path = '/content/funko-ml-main/src/evaluate.py'

print("evaluate.py exists:", os.path.exists(evaluate_path))

# Load evaluate.py directly
spec = importlib.util.spec_from_file_location(
    "evaluate",
    evaluate_path
)

evaluate = importlib.util.module_from_spec(spec)
sys.modules["evaluate"] = evaluate
spec.loader.exec_module(evaluate)

# Import required functions
from evaluate import load_data, evaluate_fn, save_results

print("All imports successful!")

evaluate.py exists: True
All imports successful!


In [11]:
d, folds = load_data()

print("X_onehot shape :", d['X_onehot'].shape)
print("X_ordinal shape:", d['X_ordinal'].shape)
print("y shape        :", d['y'].shape)
print("Number of folds:", len(folds))

X_onehot shape : (359, 59)
X_ordinal shape: (359, 4)
y shape        : (359,)
Number of folds: 4


In [12]:
from sklearn.svm import SVC
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

print("Kernel imports successful!")

Kernel imports successful!


In [13]:
WEIGHTS = np.array([2.0, 1.0, 1.0, 3.0])

# The report specifies 0 < alpha < 1,
# but does not state its numerical value.
ALPHA = 0.5

# The report uses theta in the Gaussian similarity,
# but does not state its numerical value.
THETA = 1.0

print("Weights:", WEIGHTS)
print("Alpha:", ALPHA)
print("Theta:", THETA)

Weights: [2. 1. 1. 3.]
Alpha: 0.5
Theta: 1.0


In [14]:
def h(z, alpha=ALPHA):
    z = np.clip(z, 0.0, 1.0)
    return (1.0 - z ** alpha) ** (1.0 / alpha)

In [15]:
print("h(0.0) =", h(0.0))
print("h(0.25) =", h(0.25))
print("h(0.50) =", h(0.50))
print("h(1.0) =", h(1.0))

h(0.0) = 1.0
h(0.25) = 0.25
h(0.50) = 0.08578643762690492
h(1.0) = 0.0


In [16]:
def search_volume_similarity(xi, xj, theta=THETA):
    return np.exp(
        -((xi - xj) ** 2) / (2.0 * theta ** 2)
    )

In [17]:
print(
    "Similarity between 0.2 and 0.3:",
    search_volume_similarity(0.2, 0.3)
)

print(
    "Similarity between 0.2 and 0.8:",
    search_volume_similarity(0.2, 0.8)
)

Similarity between 0.2 and 0.3: 0.9950124791926823
Similarity between 0.2 and 0.8: 0.835270211411272


In [18]:
def get_category_probabilities(values):
    values = np.asarray(values)

    unique_values, counts = np.unique(
        values,
        return_counts=True
    )

    total = len(values)

    return {
        value: count / total
        for value, count in zip(
            unique_values,
            counts
        )
    }

In [19]:
example_values = np.array([
    0, 0, 0,
    1, 1,
    2
])

example_probabilities = get_category_probabilities(
    example_values
)

print("Example probabilities:")
print(example_probabilities)

Example probabilities:
{np.int64(0): np.float64(0.5), np.int64(1): np.float64(0.3333333333333333), np.int64(2): np.float64(0.16666666666666666)}


In [20]:
def categorical_similarity(
    zi,
    zj,
    probabilities,
    alpha=ALPHA
):
    if zi == zj:
        probability = probabilities[zi]
        return h(probability, alpha)

    return 0.0

In [21]:
probabilities = {
    0: 0.5,
    1: 0.3,
    2: 0.2
}

print(
    "Same value:",
    categorical_similarity(
        0,
        0,
        probabilities
    )
)

print(
    "Different value:",
    categorical_similarity(
        0,
        1,
        probabilities
    )
)

Same value: 0.08578643762690492
Different value: 0.0


In [22]:
def custom_kernel_matrix(
    X1,
    X2,
    category_probs,
    material_probs,
    special_probs,
    alpha=ALPHA,
    theta=THETA,
    weights=WEIGHTS
):

    n1 = X1.shape[0]
    n2 = X2.shape[0]

    K = np.zeros((n1, n2))

    for i in range(n1):

        for j in range(n2):

            # U1: Search volume
            U1 = search_volume_similarity(
                X1[i, 0],
                X2[j, 0],
                theta
            )

            # U2: Category
            if X1[i, 1] == X2[j, 1]:
                U2 = h(
                    category_probs[X1[i, 1]],
                    alpha
                )
            else:
                U2 = 0.0

            # U3: Material
            if X1[i, 2] == X2[j, 2]:
                U3 = h(
                    material_probs[X1[i, 2]],
                    alpha
                )
            else:
                U3 = 0.0

            # U4: Special version
            if X1[i, 3] == X2[j, 3]:
                U4 = h(
                    special_probs[X1[i, 3]],
                    alpha
                )
            else:
                U4 = 0.0

            # Composite kernel
            K[i, j] = (
                weights[0] * U1 +
                weights[1] * U2 +
                weights[2] * U3 +
                weights[3] * U4
            )

    return K

In [23]:
X_small = d['X_ordinal'][:5]

category_probs = get_category_probabilities(
    X_small[:, 1]
)

material_probs = get_category_probabilities(
    X_small[:, 2]
)

special_probs = get_category_probabilities(
    X_small[:, 3]
)

K_small = custom_kernel_matrix(
    X_small,
    X_small,
    category_probs,
    material_probs,
    special_probs
)

print("Kernel shape:", K_small.shape)
print("\nKernel matrix:")
print(K_small)

Kernel shape: (5, 5)

Kernel matrix:
[[2.91671843 1.99994374 1.99772586 1.99975062 1.98957209]
 [1.99994374 2.91671843 1.99838455 1.99993125 1.99104257]
 [1.99772586 1.99838455 2.91671843 1.99898206 1.99702661]
 [1.99975062 1.99993125 1.99898206 2.91671843 1.99253892]
 [1.98957209 1.99104257 1.99702661 1.99253892 2.91671843]]


In [24]:
print("Is kernel symmetric?",
      np.allclose(K_small, K_small.T))

eigenvalues = np.linalg.eigvalsh(K_small)

print("Minimum eigenvalue:",
      eigenvalues.min())

print("Maximum eigenvalue:",
      eigenvalues.max())

Is kernel symmetric? True
Minimum eigenvalue: 0.9167184270002549
Maximum eigenvalue: 10.90268428229376


In [25]:
def train_custom_kernel_fold(
    X,
    y,
    train_idx,
    valid_idx,
    alpha=ALPHA,
    theta=THETA,
    weights=WEIGHTS
):

    X_train = X[train_idx]
    X_valid = X[valid_idx]

    y_train = y[train_idx]
    y_valid = y[valid_idx]

    # Probabilities from training data only
    category_probs = get_category_probabilities(
        X_train[:, 1]
    )

    material_probs = get_category_probabilities(
        X_train[:, 2]
    )

    special_probs = get_category_probabilities(
        X_train[:, 3]
    )

    # Training kernel
    K_train = custom_kernel_matrix(
        X_train,
        X_train,
        category_probs,
        material_probs,
        special_probs,
        alpha,
        theta,
        weights
    )

    # Validation-to-training kernel
    K_valid = custom_kernel_matrix(
        X_valid,
        X_train,
        category_probs,
        material_probs,
        special_probs,
        alpha,
        theta,
        weights
    )

    model = SVC(
        kernel='precomputed',
        decision_function_shape='ovo'
    )

    model.fit(
        K_train,
        y_train
    )

    train_pred = model.predict(K_train)

    valid_pred = model.predict(K_valid)

    return (
        y_train,
        train_pred,
        y_valid,
        valid_pred
    )

In [26]:
train_idx, valid_idx = folds[0]

(
    y_train_test,
    train_pred_test,
    y_valid_test,
    valid_pred_test
) = train_custom_kernel_fold(
    d['X_ordinal'],
    d['y'],
    train_idx,
    valid_idx
)

print("Fold 1 completed!")

print("Training samples:", len(y_train_test))
print("Validation samples:", len(y_valid_test))

Fold 1 completed!
Training samples: 269
Validation samples: 90


In [35]:
train_precisions = []
train_recalls = []
train_f1s = []
train_class_precisions = []

valid_precisions = []
valid_recalls = []
valid_f1s = []
valid_class_precisions = []

labels = np.unique(d['y'])

for fold_number, (train_idx, valid_idx) in enumerate(
    folds,
    start=1
):

    print(f"Running Fold {fold_number}...")

    (
        y_train_fold,
        train_pred_fold,
        y_valid_fold,
        valid_pred_fold
    ) = train_custom_kernel_fold(
        d['X_ordinal'],
        d['y'],
        train_idx,
        valid_idx
    )

    # -------------------------
    # TRAIN METRICS
    # -------------------------

    train_precisions.append(
        precision_score(
            y_train_fold,
            train_pred_fold,
            average='weighted',
            zero_division=0
        )
    )

    train_recalls.append(
        recall_score(
            y_train_fold,
            train_pred_fold,
            average='weighted',
            zero_division=0
        )
    )

    train_f1s.append(
        f1_score(
            y_train_fold,
            train_pred_fold,
            average='weighted',
            zero_division=0
        )
    )

    # Training precision for each class
    train_class_precisions.append(
        precision_score(
            y_train_fold,
            train_pred_fold,
            labels=labels,
            average=None,
            zero_division=0
        )
    )

    # -------------------------
    # VALIDATION METRICS
    # -------------------------

    valid_precisions.append(
        precision_score(
            y_valid_fold,
            valid_pred_fold,
            average='weighted',
            zero_division=0
        )
    )

    valid_recalls.append(
        recall_score(
            y_valid_fold,
            valid_pred_fold,
            average='weighted',
            zero_division=0
        )
    )

    valid_f1s.append(
        f1_score(
            y_valid_fold,
            valid_pred_fold,
            average='weighted',
            zero_division=0
        )
    )

    # Validation precision for each class
    valid_class_precisions.append(
        precision_score(
            y_valid_fold,
            valid_pred_fold,
            labels=labels,
            average=None,
            zero_division=0
        )
    )

print("\nAll 4 folds completed!")

Running Fold 1...
Running Fold 2...
Running Fold 3...
Running Fold 4...

All 4 folds completed!


In [36]:
res_custom = {
    'train': {
        'precision': np.mean(train_precisions),
        'recall': np.mean(train_recalls),
        'f1': np.mean(train_f1s)
    },

    'valid': {
        'precision': np.mean(valid_precisions),
        'recall': np.mean(valid_recalls),
        'f1': np.mean(valid_f1s)
    },

    'train_class_precision': np.mean(
        np.array(train_class_precisions),
        axis=0
    ),

    'valid_class_precision': np.mean(
        np.array(valid_class_precisions),
        axis=0
    ),

    'labels': labels
}

print("Final results created!")

Final results created!


In [37]:
print("=" * 55)
print("CUSTOM KERNEL SVM RESULTS")
print("=" * 55)

print("\nTRAIN")
print("Precision:",
      res_custom['train']['precision'])

print("Recall   :",
      res_custom['train']['recall'])

print("F1 Score :",
      res_custom['train']['f1'])

print("\nVALIDATION")
print("Precision:",
      res_custom['valid']['precision'])

print("Recall   :",
      res_custom['valid']['recall'])

print("F1 Score :",
      res_custom['valid']['f1'])

print("\nVALIDATION PRECISION BY CLASS")

for label, precision in zip(
    res_custom['labels'],
    res_custom['valid_class_precision']
):
    print(
        f"Bin {label}: {precision}"
    )

CUSTOM KERNEL SVM RESULTS

TRAIN
Precision: 0.6238886671759976
Recall   : 0.6044816191656339
F1 Score : 0.5736694083829683

VALIDATION
Precision: 0.39163071729481447
Recall   : 0.4623595505617977
F1 Score : 0.42290426293772004

VALIDATION PRECISION BY CLASS
Bin 1: 0.4819056429232192
Bin 2: 0.4590982028241335
Bin 3: 0.0
Bin 4: 0.0
Bin 5: 0.0


In [38]:
paper_comparison = pd.DataFrame({
    'Metric': [
        'Train Precision',
        'Train Recall',
        'Train F1',
        'Validation Precision',
        'Validation Recall',
        'Validation F1'
    ],

    'Paper': [
        0.662,
        0.630,
        0.602,
        0.390,
        0.460,
        0.421
    ],

    'Our Result': [
        res_custom['train']['precision'],
        res_custom['train']['recall'],
        res_custom['train']['f1'],
        res_custom['valid']['precision'],
        res_custom['valid']['recall'],
        res_custom['valid']['f1']
    ]
})

paper_comparison

,Metric,Paper,Our Result
0,Train Precision,0.662,0.623889
1,Train Recall,0.630,0.604482
2,Train F1,0.602,0.573669
3,Validation Precision,0.390,0.391631
4,Validation Recall,0.460,0.462360
5,Validation F1,0.421,0.422904


In [39]:
save_results(
    'svm_custom_kernel',
    res_custom
)

print("Custom kernel results saved successfully!")

Custom kernel results saved successfully!


In [40]:
print("=" * 60)
print("03_CUSTOM_KERNEL_SVM COMPLETED")
print("=" * 60)

print("\nModel: Custom Kernel SVM")

print("Feature representation: Ordinal Encoding")

print("Kernel weights:", WEIGHTS)

print("Alpha:", ALPHA)
print("Theta:", THETA)

print("\nDataset")
print("Samples:", len(d['y']))
print("Features:", d['X_ordinal'].shape[1])
print("Folds:", len(folds))

print("\nOur Validation Results")
print(
    "Precision:",
    round(res_custom['valid']['precision'], 4)
)

print(
    "Recall:",
    round(res_custom['valid']['recall'], 4)
)

print(
    "F1:",
    round(res_custom['valid']['f1'], 4)
)

print("\nResult file:")
print("results/svm_custom_kernel.csv")

print("\nDONE!")

03_CUSTOM_KERNEL_SVM COMPLETED

Model: Custom Kernel SVM
Feature representation: Ordinal Encoding
Kernel weights: [2. 1. 1. 3.]
Alpha: 0.5
Theta: 1.0

Dataset
Samples: 359
Features: 4
Folds: 4

Our Validation Results
Precision: 0.3916
Recall: 0.4624
F1: 0.4229

Result file:
results/svm_custom_kernel.csv

DONE!
